In [ ]:
from xml.etree import ElementTree as ET
from pathlib import Path

ns = {"alto": "http://www.loc.gov/standards/alto/ns-v4#"}

def xmltotxt(xml_path: Path, out_path: Path):
    root = ET.parse(xml_path).getroot()
    lines_out = []

    # ALTO v4, ggf. Fallback ohne Namespace
    textlines = root.findall(".//alto:TextLine", ns) or root.findall(".//{*}TextLine")

    for tl in textlines:
        parts = []
        pending_hyphen_word = None
        skip_next_hyp = False
        skip_next_hypPart2 = False

        for node in list(tl):
            tag = node.tag.split('}')[-1]

            if tag == "String":
                content = node.attrib.get("CONTENT", "")
                subs_type = node.attrib.get("SUBS_TYPE")
                subs_content = node.attrib.get("SUBS_CONTENT")

                if skip_next_hypPart2 and subs_type == "HypPart2":
                    skip_next_hypPart2 = False
                    continue

                if subs_type == "HypPart1" and subs_content:
                    pending_hyphen_word = subs_content
                    skip_next_hyp = True
                    skip_next_hypPart2 = True
                    continue

                parts.append(content)

            elif tag == "SP":
                parts.append(" ")

            elif tag == "HYP":
                if skip_next_hyp:
                    if pending_hyphen_word is not None:
                        parts.append(pending_hyphen_word)
                        pending_hyphen_word = None
                    skip_next_hyp = False
                else:
                    parts.append(node.attrib.get("CONTENT", "-"))

        if pending_hyphen_word is not None:
            parts.append(pending_hyphen_word)

        line_text = " ".join("".join(parts).split()).strip()
        if line_text:
            lines_out.append(line_text)

    out_path.write_text("\n".join(lines_out), encoding="utf-8")
    print(f"geschrieben nach: {out_path}")

# ---- rekursiv über den Ordner laufen ----
base_dir = Path(r"example_path")  # <- als Path!
for xml_file in base_dir.rglob("*.xml"):
    out_path = xml_file.with_suffix(".txt")
    xmltotxt(xml_file, out_path)


In [ ]:
from xml.etree import ElementTree as ET
from pathlib import Path
import unicodedata, re

ns = {"alto": "http://www.loc.gov/standards/alto/ns-v4#"}

# --- Normalisierung ---
TRANSLATION_MAP = {
    # Bindestriche/Trennstriche vereinheitlichen
    ord("–"): "-", ord("—"): "-", ord("-"): "-", ord("−"): "-",
    ord("⸗"): "-",  # historischer Doppelstrich

    # Anführungszeichen vereinheitlichen
    ord("„"): '"', ord("“"): '"', ord("”"): '"', ord("«"): '"', ord("»"): '"',
    ord("‚"): "'", ord("‘"): "'", ord("’"): "'",

    # Punkte/Artefakte/Ornamente entfernen
    ord("˙"): None, ord("·"): None, ord("•"): None,

    # schmale/geschützte/unsichtbare Leerzeichen
    0x00A0: 32,  # no-break space -> space
    0x202F: 32,  # narrow no-break space -> space
    0x2000: 32, 0x2001: 32, 0x2002: 32, 0x2003: 32, 0x2004: 32, 0x2005: 32,
    0x2006: 32, 0x2007: 32, 0x2008: 32, 0x2009: 32, 0x200A: 32,
    0x200B: None,  # zero-width space entfernen
    0x2060: None,  # word joiner
    0xFEFF: None,  # zero-width no-break space
}

def normalize_text(s: str) -> str:
    # 1) NFKC: macht ſ -> s, Ligaturen -> Buchstaben, Formvarianten vereinheitlicht
    s = unicodedata.normalize("NFKC", s)
    # 2) gezielte Ersetzungen/Entfernungen
    s = s.translate(TRANSLATION_MAP)
    # 3) Leerraum glätten (aber Zeilenumbrüche behalten)
    s = re.sub(r"[ \t]+", " ", s)            # Mehrfach-Spaces zu 1 Space
    s = re.sub(r"[ \t]*\n[ \t]*", "\n", s)   # Space an Zeilenenden raus
    s = s.strip()
    return s

# --- Dein ALTO -> Plaintext (Zeile für Zeile) mit Hyp-Part-Reparatur ---
def xmltotxt(xml_path: Path, out_path: Path):
    root = ET.parse(xml_path).getroot()
    textlines = root.findall(".//alto:TextLine", ns) or root.findall(".//{*}TextLine")

    lines_out = []
    for tl in textlines:
        parts = []
        pending_hyphen_word = None
        skip_next_hyp = False
        skip_next_hypPart2 = False

        for node in list(tl):
            tag = node.tag.split('}')[-1]

            if tag == "String":
                content = node.attrib.get("CONTENT", "")
                subs_type = node.attrib.get("SUBS_TYPE")
                subs_content = node.attrib.get("SUBS_CONTENT")

                if skip_next_hypPart2 and subs_type == "HypPart2":
                    skip_next_hypPart2 = False
                    continue

                if subs_type == "HypPart1" and subs_content:
                    pending_hyphen_word = subs_content  # z. B. "fraglichen"
                    skip_next_hyp = True
                    skip_next_hypPart2 = True
                    continue

                parts.append(content)

            elif tag == "SP":
                parts.append(" ")

            elif tag == "HYP":
                if skip_next_hyp:
                    if pending_hyphen_word is not None:
                        parts.append(pending_hyphen_word)
                        pending_hyphen_word = None
                    skip_next_hyp = False
                else:
                    parts.append(node.attrib.get("CONTENT", "-"))

        if pending_hyphen_word is not None:
            parts.append(pending_hyphen_word)

        raw_line = "".join(parts)
        raw_line = " ".join(raw_line.split())  # grobe Whitespace-Norm
        if raw_line:
            lines_out.append(normalize_text(raw_line))

    out_path.write_text("\n".join(lines_out), encoding="utf-8")
    print(f"geschrieben nach: {out_path}")

# --- Beispiel: rekursiv über deinen Ordner ---
base_dir = Path(r"example_path")  # <- als Path!
for xml_file in base_dir.rglob("*.xml"):
    if "fulltext" not in xml_file.parts:
        continue
    out_path = xml_file.with_suffix(".txt")  # oder .with_suffix(".norm.txt")
    xmltotxt(xml_file, out_path)


In [ ]:
from pathlib import Path
import csv, re

def classify_eras(year: int):
    # era1
    if year is None:
        era1 = ""
    elif year < 1914:
        era1 = "pre war"
    elif 1914 <= year <= 1918:
        era1 = "war"
    elif 1919 <= year <= 1920:
        era1 = "post war"
    else:
        era1 = ""  # außerhalb des von dir geforderten Bereichs

    # era2
    if year is None:
        era2 = ""
    elif year <= 1915:
        era2 = "pre"
    else:  # year >= 1916
        era2 = "post"

    return era1, era2

def page_txt_to_tsv(txt_path, out_path=None, drop_empty=True):
    """
    Erzeugt eine TSV mit Spalten:
    id, year, era1, era2, text
    - id = Prefix bis inkl. YYYY-MM-DD
    - year = Jahreszahl aus id
    - era1/era2 = Klassifikation gemäß Vorgabe
    - text = jede Textzeile als eigene Zeile
    """
    txt_path = Path(txt_path)
    if out_path is None:
        out_path = txt_path.with_suffix(".tsv")
    out_path = Path(out_path)

    # id = Prefix bis inkl. Datum (YYYY-MM-DD) aus dem Dateinamen
    # Beispiel: 3074409X_1920-01-01_000_1_H_1_001.txt -> 3074409X_1920-01-01
    m_id = re.match(r'^(?P<prefix>.+?_\d{4}-\d{2}-\d{2})_', txt_path.name)
    doc_id = m_id.group("prefix") if m_id else txt_path.stem

    # year aus dem Dateinamen extrahieren
    m_year = re.search(r'_(\d{4})-\d{2}-\d{2}', txt_path.name)
    year = int(m_year.group(1)) if m_year else None

    era1, era2 = classify_eras(year)

    with open(txt_path, "r", encoding="utf-8", errors="ignore") as fin, \
         open(out_path, "w", encoding="utf-8", newline="") as fout:
        writer = csv.DictWriter(
            fout, delimiter="\t",
            fieldnames=["id", "year", "era1", "era2", "text"],
            lineterminator="\n"
        )
        writer.writeheader()
        for line in fin.read().splitlines():
            if drop_empty and not line.strip():
                continue
            writer.writerow({
                "id": doc_id,
                "year": year if year is not None else "",
                "era1": era1,
                "era2": era2,
                "text": line.rstrip("\r\n")
            })

    print(f"TSV geschrieben: {out_path}")

#---- Beispielaufruf (Pfad anpassen) ----
page_txt_to_tsv(
    r"example_path",
    out_path=r"example_path.tsv")


In [ ]:
from pathlib import Path
import csv, re
from collections import OrderedDict, defaultdict

def read_tsv(path):
    with open(path, "r", encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        rows = [row for row in reader]
        cols = reader.fieldnames or []
    return rows, cols

def infer_emotion_from_filename(path: Path):
    # .../Joy.pt_prediction.tsv -> "Joy"
    m = re.search(r"([A-Za-z]+)\.pt_prediction\.tsv$", path.name)
    return m.group(1) if m else path.stem

def merge_emotion_tsvs(
    files, out_path,
    join_how="inner"  # "inner" oder "outer"
):
    files = [Path(f) for f in files]
    # 1) Einlesen
    tables = []
    for p in files:
        rows, cols = read_tsv(p)
        emo = infer_emotion_from_filename(p)
        tables.append({"path": p, "emotion": emo, "rows": rows, "cols": cols})

    # 2) Gemeinsame Spalten (Schnittmenge) als Join-Keys
    common = set(tables[0]["cols"])
    for t in tables[1:]:
        common &= set(t["cols"])
    key_cols = [c for c in tables[0]["cols"] if c in common]  # stabile Reihenfolge
    if not key_cols:
        raise RuntimeError("Keine gemeinsamen Spalten gefunden – bitte gleiche Schlüsselspalten in allen TSVs verwenden.")

    # 3) Für jede Tabelle: nicht-gemeinsame Spalten vorbereiten (werden mit Präfix versehen)
    per_table_noncommon_cols = []
    for t in tables:
        noncommon = [c for c in t["cols"] if c not in key_cols]
        per_table_noncommon_cols.append(noncommon)

    # 4) Index aufbauen: key -> basisdatensatz (gemeinsame Spalten) + je Emotion Werte
    # key ist ein Tupel aus den gemeinsamen Spaltenwerten
    def key_of(row):
        return tuple(row.get(k, "") for k in key_cols)

    # Wenn outer: sammle Keys aus allen Tabellen; wenn inner: nur Schnittmenge
    key_sets = []
    for t in tables:
        key_sets.append({key_of(r) for r in t["rows"]})
    if join_how == "outer":
        all_keys = set().union(*key_sets)
    else:
        all_keys = set(key_sets[0])
        for ks in key_sets[1:]:
            all_keys &= ks

    # Basisdaten pro Key aus der ersten Tabelle, falls vorhanden; sonst aus der nächsten, die ihn hat
    base_rows = {}
    for k in all_keys:
        base_rows[k] = OrderedDict((c, "") for c in key_cols)

    for t in tables:
        for r in t["rows"]:
            k = key_of(r)
            if k in base_rows:
                for c in key_cols:
                    if not base_rows[k][c]:
                        base_rows[k][c] = r.get(c, "")

    # 5) Emotionsspalten aufbauen und befüllen
    # Spaltenheader in stabiler Reihenfolge: key_cols + (pro Tabelle ihre noncommon-Spalten mit Präfix)
    header = list(key_cols)
    for t, noncommon in zip(tables, per_table_noncommon_cols):
        for c in noncommon:
            header.append(f"{t['emotion']}_{c}")

    # Wertecontainer initialisieren
    merged_rows = []
    # Vorab default dicts für schnelleren Zugriff je Tabelle
    per_table_index = []
    for t in tables:
        idx = {}
        for r in t["rows"]:
            idx[key_of(r)] = r
        per_table_index.append((t, idx))

    for k in sorted(all_keys):
        row_out = OrderedDict()
        # gemeinsame Spalten
        for c in key_cols:
            row_out[c] = base_rows[k].get(c, "")
        # emotionsspezifisch
        for (t, idx), noncommon in zip(per_table_index, per_table_noncommon_cols):
            r = idx.get(k)
            for c in noncommon:
                out_c = f"{t['emotion']}_{c}"
                row_out[out_c] = r.get(c, "") if r else ""
        merged_rows.append(row_out)

    # 6) Schreiben
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=header, delimiter="\t", lineterminator="\n")
        w.writeheader()
        for r in merged_rows:
            w.writerow(r)

    print(f"✔ Zusammengeführt: {len(merged_rows)} Zeilen, {len(header)} Spalten → {out_path}")
    return out_path

# ---------- Beispielaufruf ----------
base = Path(r"example_path")  # <- anpassen
files = [
    base / "Agitation.pt_prediction.tsv",
    base / "Anger.pt_prediction.tsv",
    base / "Fear.pt_prediction.tsv",
    base / "Joy.pt_prediction.tsv",
    base / "Love.pt_prediction.tsv",
    base / "Sadness.pt_prediction.tsv",
]
out = base / "ALL_emotions_merged.tsv"
merge_emotion_tsvs(files, out_path=out, join_how="inner")  # oder join_how="outer"


In [ ]:
from pathlib import Path
import csv, re

# --------- Era-Logik wie bei dir ----------
def classify_eras(year: int):
    if year is None:
        era1, era2 = "", ""
    elif year < 1914:
        era1, era2 = "pre war", ("pre" if year <= 1915 else "post")
    elif 1914 <= year <= 1918:
        era1, era2 = "war", ("pre" if year <= 1915 else "post")
    elif 1919 <= year <= 1920:
        era1, era2 = "post war", ("pre" if year <= 1915 else "post")
    else:
        era1, era2 = "", ("pre" if year <= 1915 else "post")
    return era1, era2

# --------- Metadaten aus Dateinamen ziehen ----------
DATE_RE = re.compile(r'_(\d{4})-(\d{2})-(\d{2})_')  # _YYYY-MM-DD_

def parse_meta(txt_path: Path):
    """
    Gibt (doc_id, year:int|None, month:int|None, day:int|None) zurück.
    doc_id = Prefix bis inkl. YYYY-MM-DD.
    """
    name = txt_path.name
    m_id = re.match(r'^(?P<prefix>.+?_\d{4}-\d{2}-\d{2})_', name)
    doc_id = m_id.group("prefix") if m_id else txt_path.stem

    m = DATE_RE.search(name)
    if not m:
        return doc_id, None, None, None
    y, mo, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
    return doc_id, y, mo, d

# --------- Eine Seite (erste Seite eines Monatsersten) anhängen ----------
def append_page_if_month_first(writer, txt_path: Path, author_value="nn", drop_empty=True):
    doc_id, year, month, day = parse_meta(txt_path)
    if day != 1:
        return False  # kein Monatserster

    era1, era2 = classify_eras(year)
    with txt_path.open("r", encoding="utf-8", errors="ignore") as fin:
        for line in fin.read().splitlines():
            if drop_empty and not line.strip():
                continue
            writer.writerow({
                "id": doc_id,
                "author": author_value,
                "year": year if year is not None else "",
                "era1": era1,
                "era2": era2,
                "text": line.rstrip("\r\n")
            })
    return True

# --------- Hauptfunktion: nur Monatserste, erste Seite -> eine gemeinsame TSV ----------
def month_first_firstpages_to_one_tsv(base_dir, out_tsv,
                                      author_value="nn", drop_empty=True):
    """
    Durchsucht rekursiv `base_dir` nach 'fulltext'-Ordnern, nimmt pro Ausgabe
    Dateien mit Suffix '_001.txt' (erste Seite) und schreibt NUR die,
    deren Datum den Tag '01' hat (Monatserster), in eine gemeinsame TSV.
    """
    base = Path(base_dir)
    out = Path(out_tsv)
    out.parent.mkdir(parents=True, exist_ok=True)

    # für stabile Reihenfolge: erst alle Kandidaten einsammeln und nach Datum sortieren
    candidates = []
    for fulltext_dir in base.rglob("fulltext"):
        for p in fulltext_dir.glob("*.txt"):
            if not p.name.endswith("_001.txt"):
                continue
            # nur Files mit erkennbarer Datumsstruktur
            if DATE_RE.search(p.name):
                candidates.append(p)

    # sortiere nach (year, month, day, Pfad), damit die Ausgabe deterministisch ist
    def sort_key(p: Path):
        _, y, mo, d = parse_meta(p)
        y = y or 0; mo = mo or 0; d = d or 0
        return (y, mo, d, str(p))
    candidates.sort(key=sort_key)

    processed = 0
    with out.open("w", encoding="utf-8", newline="") as fout:
        writer = csv.DictWriter(
            fout, delimiter="\t",
            fieldnames=["id", "author", "year", "era1", "era2", "text"],
            lineterminator="\n"
        )
        writer.writeheader()

        for p in candidates:
            ok = append_page_if_month_first(writer, p, author_value=author_value, drop_empty=drop_empty)
            if ok:
                processed += 1
                print(f"[OK] Monatserster: {p}")

    print(f"\nFertig: {processed} erste Seiten (Monatserste) → {out}")

# --------- Beispielaufruf (Pfade anpassen) ---------
month_first_firstpages_to_one_tsv(
    base_dir=r"example_path",
    out_tsv=r"example_file",
    author_value="nn",
    drop_empty=True
)


In [ ]:
from pathlib import Path
import csv, re, unicodedata

# ---------------- Normalisierung ----------------
TRANSLATION_MAP = {
    # Bindestriche vereinheitlichen
    ord("–"): "-", ord("—"): "-", ord("−"): "-", 0x00AD: None,   # Soft Hyphen
    # Anführungen vereinheitlichen
    ord("„"): '"', ord("“"): '"', ord("”"): '"', ord("«"): '"', ord("»"): '"',
    ord("‚"): "'", ord("‘"): "'", ord("’"): "'",
    # geschützte/schmale/unsichtbare Spaces
    0x00A0: 32, 0x202F: 32, 0x2000: 32, 0x2001: 32, 0x2002: 32, 0x2003: 32,
    0x2004: 32, 0x2005: 32, 0x2006: 32, 0x2007: 32, 0x2008: 32, 0x2009: 32,
    0x200A: 32, 0x200B: None, 0x200C: None, 0x200D: None, 0x2060: None, 0xFEFF: None,
    # typische OCR-Artefakte/Bullets/Kästen
    ord("•"): None, ord("▪"): None, ord("‣"): None, ord("◦"): None,
    ord("■"): None, ord("◼"): None, ord("▮"): None, ord("☞"): None,
}

def _strip_controls(s: str) -> str:
    # Entfernt alle Unicode-Control/Format-Zeichen; \n und \t bleiben erhalten
    return "".join(ch for ch in s if ch in "\n\t" or unicodedata.category(ch)[0] != "C")

def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFKC", s)
    s = s.translate(TRANSLATION_MAP)
    s = _strip_controls(s)
    # Leerraum glätten (Zeilenumbrüche erhalten)
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"[ \t]*\n[ \t]*", "\n", s)
    return s.strip()

# Optional strengere Whitelist (nur bei Bedarf auf True setzen)
ALLOW = re.compile(r"[^A-Za-zÄÖÜäöüß0-9 .,:;!?()\"'\/\-]\n?")
def strict_whitelist(s: str) -> str:
    # hält deutsche Buchstaben, Ziffern, Standardzeichen; \n bleibt erhalten
    return re.sub(r"[^\n]", lambda m: "" if ALLOW.match(m.group(0)) else m.group(0), s)

# ---------------- Meta / Epochen ----------------
def classify_eras(year: int):
    if year is None:
        return "", ""
    era1 = "pre war" if year < 1914 else ("war" if year <= 1918 else ("post war" if year <= 1920 else ""))
    era2 = "pre" if year <= 1915 else "post"
    return era1, era2

DATE_RE = re.compile(r'_(\d{4})-(\d{2})-(\d{2})_')  # _YYYY-MM-DD_

def parse_meta(txt_path: Path):
    """
    -> (doc_id, year:int|None, month:int|None, day:int|None)
    doc_id = Prefix bis inkl. YYYY-MM-DD
    """
    name = txt_path.name
    m_id = re.match(r'^(?P<prefix>.+?_\d{4}-\d{2}-\d{2})_', name)
    doc_id = m_id.group("prefix") if m_id else txt_path.stem
    m = DATE_RE.search(name)
    if not m:
        return doc_id, None, None, None
    return doc_id, int(m.group(1)), int(m.group(2)), int(m.group(3))

# ---------------- Schreiben einer Seite ----------------
def append_page_if_month_first(writer, txt_path: Path, author_value="nn", drop_empty=True, strict=False):
    doc_id, year, month, day = parse_meta(txt_path)
    if day != 1:
        return False  # nur Monatserster
    era1, era2 = classify_eras(year)

    with txt_path.open("r", encoding="utf-8", errors="ignore") as fin:
        for raw in fin.read().splitlines():
            line = normalize_text(raw)
            if strict:
                line = strict_whitelist(line)
            if drop_empty and not line:
                continue
            writer.writerow({
                "id": doc_id,
                "author": author_value,
                "year": year if year is not None else "",
                "era1": era1,
                "era2": era2,
                "text": line
            })
    return True

# ---------------- Hauptfunktion ----------------
def month_first_firstpages_to_one_tsv(base_dir, out_tsv,
                                      author_value="nn", drop_empty=True,
                                      first_page_suffix="_001.txt", strict=False):
    """
    Sucht rekursiv ab base_dir nach 'fulltext', nimmt nur erste Seiten (Suffix _001.txt)
    vom Monatsersten (Tag=01) und schreibt alles in eine gemeinsame TSV.
    """
    base = Path(base_dir)
    out = Path(out_tsv)
    out.parent.mkdir(parents=True, exist_ok=True)

    # Kandidaten einsammeln (nur _001.txt mit Datum im Namen)
    candidates = []
    for fulltext_dir in base.rglob("fulltext"):
        for p in fulltext_dir.glob(f"*{first_page_suffix}"):
            if DATE_RE.search(p.name):
                candidates.append(p)

    # deterministische Reihenfolge: nach Datum sortieren
    def sort_key(p: Path):
        _, y, mo, d = parse_meta(p)
        return (y or 0, mo or 0, d or 0, str(p))
    candidates.sort(key=sort_key)

    processed = 0
    with out.open("w", encoding="utf-8", newline="") as fout:
        writer = csv.DictWriter(
            fout, delimiter="\t",
            fieldnames=["id", "author", "year", "era1", "era2", "text"],
            lineterminator="\n"
        )
        writer.writeheader()
        for p in candidates:
            if append_page_if_month_first(writer, p, author_value=author_value,
                                          drop_empty=drop_empty, strict=strict):
                processed += 1
                print(f"[OK] Monatserster: {p}")

    print(f"\nFertig: {processed} erste Seiten (Monatserste) → {out}")

# ---------------- Beispielaufruf (Pfade anpassen) ----------------
month_first_firstpages_to_one_tsv(
    base_dir=r"example_path",
    out_tsv=r"example_file",
    author_value="nn",
    drop_empty=True,
    first_page_suffix="_001.txt",  # falls nötig auf "_000.txt" ändern
    strict=False                   # auf True setzen, wenn Whitelist zusätzlich greifen soll
)


In [ ]:
from pathlib import Path
import csv, tempfile, os

def slim_tsv(in_tsv, out_tsv=None, keep=("id","text"), in_place=False):
    """
    Behalte nur Spalten in `keep` (Standard: id, text).
    - in_place=True: Originaldatei wird atomar ersetzt (Windows-sicher, FD wird geschlossen).
    - out_tsv: Zielpfad (nur wenn in_place=False).
    """
    in_tsv = Path(in_tsv)

    if in_place:
        # Sichere, geschlossene Temp-Datei erstellen (FD schließen!)
        fd, tmp_path = tempfile.mkstemp(prefix=in_tsv.stem + "_", suffix=".tmp", dir=str(in_tsv.parent))
        os.close(fd)
        tmp = Path(tmp_path)
        out_path = tmp
    else:
        out_path = Path(out_tsv) if out_tsv else in_tsv.with_suffix(".slim.tsv")

    # Schreiben (nur gewünschte Spalten)
    with in_tsv.open("r", encoding="utf-8", errors="ignore", newline="") as fin, \
         out_path.open("w", encoding="utf-8", newline="") as fout:
        r = csv.DictReader(fin, delimiter="\t")
        cols = [c for c in keep if c in (r.fieldnames or [])]
        if not cols:
            raise ValueError(f"Keine der gewünschten Spalten {keep} in {in_tsv.name} gefunden.")
        w = csv.DictWriter(fout, fieldnames=cols, delimiter="\t", lineterminator="\n")
        w.writeheader()
        for row in r:
            w.writerow({k: row.get(k, "") for k in cols})

    if in_place:
        # Original ersetzen (stellt sicher, dass keine Datei mehr offen ist)
        backup = in_tsv.with_suffix(in_tsv.suffix + ".bak")
        try:
            if backup.exists():
                backup.unlink()
            os.replace(in_tsv, backup)   # Original → Backup
            os.replace(out_path, in_tsv) # tmp → Originalname
            backup.unlink(missing_ok=True)
            print(f"[OK] in-place geslimmt: {in_tsv}")
        except Exception as e:
            # Aufräumen & Wiederherstellen
            try:
                if out_path.exists():
                    out_path.unlink(missing_ok=True)
                if backup.exists():
                    os.replace(backup, in_tsv)
            finally:
                raise e
    else:
        print(f"[OK] geschrieben: {out_path}")

# --- Beispiel: einzelne Datei in-place (nur id & text behalten) ---
# ACHTUNG: Datei darf nicht in Excel/Editor geöffnet sein!
slim_tsv(r"example_path", in_place=True)

# --- Beispiel: ohne In-Place (neue Datei .slim.tsv neben der alten) ---
# slim_tsv(r"C:\Users\sam97xs\Stabi_Hackathon\inputfile_tag.tsv", in_place=False)


In [ ]:
from pathlib import Path
import csv, os, tempfile, re

def _to_int_year(v):
    """Robustes Year-Parsing: akzeptiert '1914', '1914.0', oder eingebettet im String."""
    s = str(v).strip()
    if not s:
        return None
    # reine Ganzzahl?
    if re.fullmatch(r"\d{4}", s):
        return int(s)
    # float-artig?
    try:
        f = float(s.replace(",", "."))
        if 1800 <= f <= 2100:
            return int(f)
    except Exception:
        pass
    # irgendwo 4-stellige Jahreszahl?
    m = re.search(r"\b(18|19|20)\d{2}\b", s)
    return int(m.group(0)) if m else None

def fix_era1_for_year(
    in_tsv,
    target_year=1914,
    era1_col="era1",
    year_col="year",
    new_value="pre war",
    only_if_current=None,   # z.B. "war"; None = immer setzen, sobald year passt
    in_place=True,
    out_tsv=None
):
    """
    Setzt in allen Zeilen mit year==target_year die Spalte `era1` auf `new_value`.
    - only_if_current: wenn z.B. "war", dann nur ändern, falls era1 bisher genau "war" ist.
    - in_place=True: Eingabedatei wird atomar ersetzt (Windows-sicher).
    """
    in_tsv = Path(in_tsv)

    # Temp-Datei anlegen (FD sofort schließen -> wichtig für Windows)
    if in_place:
        fd, tmp_path = tempfile.mkstemp(prefix=in_tsv.stem + "_", suffix=".tmp", dir=str(in_tsv.parent))
        os.close(fd)
        out_path = Path(tmp_path)
    else:
        out_path = Path(out_tsv) if out_tsv else in_tsv.with_suffix(".fixed.tsv")

    changed = total = 0
    with in_tsv.open("r", encoding="utf-8", errors="ignore", newline="") as fin, \
         out_path.open("w", encoding="utf-8", newline="") as fout:
        r = csv.DictReader(fin, delimiter="\t")
        fieldnames = r.fieldnames or []
        if year_col not in fieldnames or era1_col not in fieldnames:
            raise ValueError(f"Spalten '{year_col}' und/oder '{era1_col}' fehlen. Vorhanden: {fieldnames}")
        w = csv.DictWriter(fout, fieldnames=fieldnames, delimiter="\t", lineterminator="\n")
        w.writeheader()

        for row in r:
            total += 1
            y = _to_int_year(row.get(year_col, ""))
            if y == target_year:
                if (only_if_current is None) or (row.get(era1_col, "").strip().lower() == only_if_current.lower()):
                    row[era1_col] = new_value
                    changed += 1
            w.writerow(row)

    if in_place:
        os.replace(out_path, in_tsv)
        print(f"[OK] In-place aktualisiert: {in_tsv}")
    else:
        print(f"[OK] geschrieben: {out_path}")

    print(f"Geprüft: {total} Zeilen | Geändert (year=={target_year}): {changed}")

# ==== Beispielaufrufe (Pfad anpassen) ====
# In-place auf deiner Datei:
fix_era1_for_year(r"example_file")



In [ ]:
from pathlib import Path
import csv, os, re, tempfile

# ---- Jahr aus id extrahieren ----
DATE_RE = re.compile(r'_(\d{4})-(\d{2})-(\d{2})(?:_|$)')
FALLBACK_YEAR = re.compile(r'\b(18\d{2}|19\d{2}|20\d{2})\b')

def extract_year(id_value: str):
    s = str(id_value or "")
    m = DATE_RE.search(s)
    if m:
        return int(m.group(1))
    m = FALLBACK_YEAR.search(s)
    return int(m.group(1)) if m else None

def classify_eras(year: int):
    # era1
    if year is None:
        era1 = ""
    elif year < 1914:
        era1 = "pre war"
    elif 1914 <= year <= 1918:
        era1 = "war"
    elif 1919 <= year <= 1920:
        era1 = "post war"
    else:
        era1 = ""
    # era2
    if year is None:
        era2 = ""
    elif year <= 1915:
        era2 = "pre"
    else:
        era2 = "post"
    return era1, era2

def add_year_eras(in_tsv, out_tsv=None, in_place=True,
                  id_col="id", year_col="year", era1_col="era1", era2_col="era2"):
    """
    Ergänzt year/era1/era2. Spalten bleiben bestehen; falls year/era* schon existieren,
    werden sie überschrieben.
    """
    in_tsv = Path(in_tsv)

    # sichere Temp-Datei (Windows: FD gleich schließen)
    if in_place:
        fd, tmp_path = tempfile.mkstemp(prefix=in_tsv.stem + "_", suffix=".tmp", dir=str(in_tsv.parent))
        os.close(fd)
        out_path = Path(tmp_path)
    else:
        out_path = Path(out_tsv) if out_tsv else in_tsv.with_suffix(".with_eras.tsv")

    with in_tsv.open("r", encoding="utf-8", errors="ignore", newline="") as fin:
        r = csv.DictReader(fin, delimiter="\t")
        if id_col not in (r.fieldnames or []):
            raise ValueError(f"Spalte '{id_col}' nicht gefunden. Vorhanden: {r.fieldnames}")

        # neue Headerliste: vorhandene + (year, era1, era2) ans Ende (ohne Duplikate)
        fieldnames = list(r.fieldnames)
        for c in (year_col, era1_col, era2_col):
            if c not in fieldnames:
                fieldnames.append(c)

        with out_path.open("w", encoding="utf-8", newline="") as fout:
            w = csv.DictWriter(fout, fieldnames=fieldnames, delimiter="\t", lineterminator="\n")
            w.writeheader()

            for row in r:
                y = extract_year(row.get(id_col, ""))
                era1, era2 = classify_eras(y)
                row[year_col] = y if y is not None else ""
                row[era1_col] = era1
                row[era2_col] = era2
                w.writerow(row)

    if in_place:
        os.replace(out_path, in_tsv)
        print(f"[OK] in-place aktualisiert: {in_tsv}")
    else:
        print(f"[OK] geschrieben: {out_path}")

# ===== Beispielaufrufe =====
# 1) Einzeldatei in-place (empfohlen, wenn Datei nicht in Excel/Editor geöffnet ist):
add_year_eras(r"example_path", in_place=True)




In [ ]:
# Monatsmittel (absolut) + 3-Monats-Glättung + MWU-Tests auf Monatsaggregaten (ohne NumPy/Pandas)
import sys, subprocess, csv, re, math, datetime as dt
from collections import defaultdict, OrderedDict

# Plotly installieren/aktivieren
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"], check=False)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === Pfad anpassen ===
TSV = r"example_path"

EMOS = ["Joy","Love","Fear","Anger","Sadness","Agitation"]

def to_float(x):
    if x is None: return 0.0
    s = str(x).strip()
    if not s: return 0.0
    try: return float(s.replace(",", "."))
    except: return 0.0

DATE_RE = re.compile(r"(\d{4})-(\d{2})-(\d{2})")
def parse_ymd_from_id(s):
    m = DATE_RE.search(s or "")
    if not m: return None
    return int(m.group(1)), int(m.group(2)), int(m.group(3))

def classify_era1(year):
    if year is None: return ""
    if year < 1914: return "pre war"
    if 1914 <= year <= 1918: return "war"
    if 1919 <= year <= 1920: return "post war"
    return ""

def classify_era2(year):
    if year is None: return ""
    return "pre" if year <= 1915 else "post"

# --- 1) Monatsmittel aggregieren ---
# sums[(y,m)][emo], cnts[(y,m)]
sums = defaultdict(lambda: defaultdict(float))
cnts = defaultdict(int)

with open(TSV, "r", encoding="utf-8", errors="ignore", newline="") as f:
    r = csv.DictReader(f, delimiter="\t")
    headers = r.fieldnames or []
    need = [f"mean_{e}" for e in EMOS] + ["id"]
    missing = [c for c in need if c not in headers]
    if missing:
        raise RuntimeError(f"Fehlende Spalten: {missing}\nGefunden: {headers}")
    for row in r:
        ymd = parse_ymd_from_id(row.get("id",""))
        if not ymd: 
            continue
        y, m, d = ymd
        key = (y, m)
        cnts[key] += 1
        for emo in EMOS:
            sums[key][emo] += to_float(row.get(f"mean_{emo}", ""))

# Monatsmittel (absolute Werte)
ym_keys = sorted(cnts.keys())  # chronologisch
monthly_means = OrderedDict()
for ym in ym_keys:
    n = max(1, cnts[ym])
    monthly_means[ym] = {e: sums[ym][e] / n for e in EMOS}

x_dates = [dt.date(y, m, 1) for (y, m) in ym_keys]

# --- 2) 3-Monats-Glättung ---
def moving_avg(vals, win=3):
    if not vals: return []
    k = win//2
    out = []
    for i in range(len(vals)):
        lo = max(0, i-k)
        hi = min(len(vals), i+k+1)
        denom = hi - lo
        out.append(sum(vals[lo:hi]) / denom if denom>0 else vals[i])
    return out

# Farben (Positiv grün, Negativ rot)
pos_palette = {"Joy":"#2ca25f","Love":"#66c2a4"}
neg_palette = {"Anger":"#b2182b","Fear":"#d6604d","Sadness":"#f4a582","Agitation":"#d7301f"}
color_map = {**pos_palette, **neg_palette}

# --- 3) Linienplots: Original + geglättet ---
fig = go.Figure()
for emo in EMOS:
    y_raw = [monthly_means[ym][emo] for ym in ym_keys]
    y_smooth = moving_avg(y_raw, win=3)
    c = color_map.get(emo, None)
    fig.add_trace(go.Scatter(x=x_dates, y=y_raw, name=f"{emo} (Monat)",
                             mode="lines", line=dict(width=1, color=c), opacity=0.5))
    fig.add_trace(go.Scatter(x=x_dates, y=y_smooth, name=f"{emo} (3M-Glättung)",
                             mode="lines", line=dict(width=3, color=c)))
fig.update_layout(
    title="Emotionen – absolute Monatsmittel (inkl. 3-Monats-Glättung)",
    template="plotly_white", hovermode="x unified",
    xaxis_title="Monat", yaxis_title="Monatsmittel (absolut)"
)
fig.update_xaxes(dtick="M3", tickformat="%Y-%m")
fig.show()

# --- 4) MWU-Tests auf Monatsaggregaten (era2 & era1) ---
# Hilfsfunktionen: MWU, BH-FDR
def _rankdata(values):
    pairs = sorted((v,i) for i,v in enumerate(values))
    ranks = [0.0]*len(values)
    i=0
    while i<len(pairs):
        j=i+1
        while j<len(pairs) and pairs[j][0]==pairs[i][0]:
            j+=1
        avg=(i+1+j)/2.0
        for k in range(i,j):
            ranks[pairs[k][1]] = avg
        i=j
    return ranks

def cliffs_delta(a,b):
    if not a or not b: return 0.0
    a_sorted = sorted(a); b_sorted = sorted(b)
    n1=len(a_sorted); n2=len(b_sorted)
    j=0; wins=0; ties=0
    for av in a_sorted:
        while j<n2 and b_sorted[j]<av: j+=1
        k=j
        while k<n2 and b_sorted[k]==av: k+=1
        wins += j; ties += (k-j)
    total=n1*n2
    losses = total - wins - ties
    return (wins - losses)/total if total else 0.0

def mw_u(x,y):
    n1,n2=len(x),len(y)
    if n1==0 or n2==0: 
        return {"p":1.0,"z":0.0,"delta":0.0,"n1":n1,"n2":n2}
    allv = x+y
    ranks = _rankdata(allv)
    R1 = sum(ranks[:n1])
    U1 = R1 - n1*(n1+1)/2.0
    N=n1+n2
    sp=sorted(allv); T=0.0; i=0
    while i<N:
        j=i+1
        while j<N and sp[j]==sp[i]:
            j+=1
        t=j-i
        if t>1: T += t**3 - t
        i=j
    meanU = n1*n2/2.0
    sigma_sq = (n1*n2/12.0) * (N+1 - T/(N*(N-1))) if N>1 else 0.0
    sigma = math.sqrt(max(sigma_sq, 1e-12))
    if U1>meanU: z=(U1-meanU-0.5)/sigma
    elif U1<meanU: z=(U1-meanU+0.5)/sigma
    else: z=0.0
    p = 2.0*(1.0 - 0.5*(1.0 + math.erf(abs(z)/math.sqrt(2))))
    return {"p":p,"z":z,"delta":cliffs_delta(x,y),"n1":n1,"n2":n2}

def bh_fdr(pairs):  # [(key,p)]
    m=len(pairs)
    if m==0: return {}
    ranked = sorted(pairs, key=lambda kv: kv[1])
    qs=[0.0]*m
    for i,(_,p) in enumerate(ranked, start=1):
        qs[i-1] = p*m / i
    # monotone
    for i in range(m-2, -1, -1):
        qs[i] = min(qs[i], qs[i+1])
    return {ranked[i][0]: min(1.0, qs[i]) for i in range(m)}

# Monatswerte je Emotion in Gruppen (era1/era2)
emo_era2 = {emo: {"pre":[], "post":[]} for emo in EMOS}
emo_era1 = {emo: {"pre war":[], "war":[], "post war":[]} for emo in EMOS}

for (y,m) in ym_keys:
    e1 = classify_era1(y)
    e2 = classify_era2(y)
    for emo in EMOS:
        v = monthly_means[(y,m)][emo]
        if e2: emo_era2[emo][e2].append(v)
        if e1: emo_era1[emo][e1].append(v)

# era2: pre vs post
era2_stats = {}
era2_pvals = []
for emo in EMOS:
    s = mw_u(emo_era2[emo]["pre"], emo_era2[emo]["post"])
    era2_stats[emo] = s
    era2_pvals.append((emo, s["p"]))
era2_q = bh_fdr(era2_pvals)

# era1: beste Paarung (min q)
pairs_era1 = [("pre war","war"),("war","post war"),("pre war","post war")]
era1_pair_stats = {}
era1_pvals = []
for emo in EMOS:
    for a,b in pairs_era1:
        s = mw_u(emo_era1[emo][a], emo_era1[emo][b])
        era1_pair_stats[(emo,(a,b))] = s
        era1_pvals.append(((emo,(a,b)), s["p"]))
era1_q_all = bh_fdr(era1_pvals)
era1_best = {}
for emo in EMOS:
    best=None
    for pair in pairs_era1:
        q = era1_q_all.get((emo,pair),1.0)
        if best is None or q < best["q"]:
            s = era1_pair_stats[(emo,pair)]
            best={"pair":pair,"p":s["p"],"q":q,"z":s["z"],"delta":s["delta"]}
    era1_best[emo]=best

# --- 5) Vergleichsplot −log10(q) ---
def nlog10(p): 
    p = max(min(p, 1.0), 1e-300); 
    return -math.log10(p)

emotions = EMOS
era2_bars = [nlog10(era2_q.get(e,1.0)) for e in emotions]
era1_bars = [nlog10(era1_best[e]["q"]) for e in emotions]

cmp = go.Figure()
cmp.add_bar(name="era1 (beste Paarung, FDR)", x=emotions, y=era1_bars, marker_color="#9ecae1")
cmp.add_bar(name="era2 (pre vs post, FDR)", x=emotions, y=era2_bars, marker_color="#fb9a99")
cmp.add_hline(y=nlog10(0.05), line_dash="dash", line_color="gray",
              annotation_text="q=0.05", annotation_position="top right", opacity=0.6)
cmp.add_hline(y=nlog10(0.01), line_dash="dot", line_color="gray",
              annotation_text="q=0.01", annotation_position="bottom right", opacity=0.6)
cmp.update_layout(
    title="-log10(FDR-q) auf Monatsaggregaten: era1 (beste Paarung) vs. era2",
    template="plotly_white", barmode="group",
    yaxis_title="-log10(q) (höher = stärkere Evidenz)", xaxis_title="Emotion")
cmp.show()

# --- 6) Kurzreport in der Konsole ---
print("\n=== ERA2 (Monatsaggregate) – pre vs post ===")
for e in EMOS:
    s = era2_stats[e]
    print(f"{e:10s} n={s['n1']+s['n2']:3d} pre={s['n1']:2d} post={s['n2']:2d}  p={s['p']:.3g} q={era2_q.get(e,1.0):.3g}  z={s['z']:.2f}  δ={s['delta']:.3f}")

print("\n=== ERA1 (Monatsaggregate) – beste Paarung pro Emotion ===")
for e in EMOS:
    b = era1_best[e]; a,bp = b["pair"]
    print(f"{e:10s} best={a} vs {bp:9s}  p={b['p']:.3g} q={b['q']:.3g}  z={b['z']:.2f}  δ={b['delta']:.3f}")
